## Downloading and preprocess the corpus

In [489]:
import nltk
nltk.download("brown")
nltk.download("webtext")
nltk.download("reuters")
nltk.download("punkt_tab")
from nltk.corpus import brown, webtext, reuters
brown_corpus = brown.sents()
brown_corpus = [" ".join(sentence) for sentence in brown_corpus]
brown_corpus = ["<s> " + sentence + " </s>" for sentence in brown_corpus][:5000]
webtext_corpus = webtext.sents()
webtext_corpus = [" ".join(sentence) for sentence in webtext_corpus]
webtext_corpus = ["<s> " + sentence + " </s>" for sentence in webtext_corpus][:5000]
reuters_corpus = reuters.sents()
reuters_corpus = [" ".join(sentence) for sentence in reuters_corpus]
reuters_corpus = ["<s> " + sentence + " </s>" for sentence in reuters_corpus][:5000]

[nltk_data] Downloading package brown to /Users/mukund/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package webtext to /Users/mukund/nltk_data...
[nltk_data]   Package webtext is already up-to-date!
[nltk_data] Downloading package reuters to /Users/mukund/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mukund/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
from collections import Counter
import string
translator = str.maketrans('', '', string.punctuation)

## Q1

In [12]:
brown_corpus[0]

"<s> The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . </s>"

In [43]:
def calc_unigram(corpus):
    all_words=[]
    for sentence in corpus:
        sentence=sentence.replace('<s>', '').replace('</s>', '')
        sentence=sentence.lower()
        sentence=sentence.translate(translator)
        words=sentence.split()
        for word in words:
            all_words.append(word)

    word_counts = Counter(all_words)
    total_words=len(word_counts)

    probabilities={word : count/total_words for word, count in word_counts.items()}

    
    return probabilities

def get_unigram(word, corpus):
    probabilities=calc_unigram(corpus)
    return probabilities.get(word)

In [47]:
get_unigram("the", brown_corpus)

0.5184355874011046

## Q2

In [52]:
brown_corpus[0]

"<s> The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place . </s>"

### Part (i)

In [830]:
all_words=[]

In [823]:
def get_bigrams(corpus):
    bigrams=[]
    for sentence in corpus:
        words=sentence.split()
        for word in words:
            all_words.append(word)
    bigrams=list(zip(all_words[:-1], all_words[1:]))
    return bigrams

In [407]:
get_bigrams(["is the first time this week in your house"])

[('is', 'the'),
 ('the', 'first'),
 ('first', 'time'),
 ('time', 'this'),
 ('this', 'week'),
 ('week', 'in'),
 ('in', 'your'),
 ('your', 'house')]

In [825]:
bigrams=get_bigrams(brown_corpus)

### Part ii

In [828]:
# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

15086
118731


In [418]:
def calc_bigram(word1, word2, bigrams):
    occurances=bigrams.count((word1, word2))
    N_w1=Counter(word1 for word1, _ in bigrams)
    w1=N_w1[word1]
    return (occurances+1)/(w1+V)

In [420]:
def calc_prob1(bigrams, word1, word2):
    prob=calc_bigram(word1, word2, bigrams)
    return prob

In [214]:
calc_prob1(bigrams, "This", "is")

0.0016175122661346848

### Part iii

In [216]:
def predict_next(word1, bigrams):
    b_counter=Counter(bigrams)
    N_w1=Counter(word1 for word1, _ in bigrams)
    w1=N_w1[word1]
    pred=None
    max_p=0
    for word2 in vocab:
        c_w1w2=b_counter[(word1, word2)]
        p=(c_w1w2+1)/(V+w1)
        if p > max_p:
            max_p=p
            pred=word2
    return pred

In [218]:
predict_next("this", bigrams)

'year'

### Part iv

In [227]:
def predict_sentence(initial, bigrams, limit=5):
    words=initial.split()
    for i in range(limit-len(words)):
        next_word=predict_next(words[-1], bigrams)
        if not next_word:
            break
        words.append(next_word)
    return " ".join(words)

In [438]:
predict_sentence("<s> in this light", bigrams, limit=10)

'<s> in this light of the first time . </s>'

## Q3

### Part i

In [440]:
all_words=[]

In [442]:
def get_trigrams(corpus):
    trigrams=[]
    for sentence in corpus:
        words=sentence.split()
        for word in words:
            all_words.append(word)
    trigrams=list(zip(all_words[:-2], all_words[1:-1], all_words[2:]))
    return trigrams

In [444]:
trigrams=get_trigrams(brown_corpus)

In [288]:
trigrams

[('The', 'quick', 'brown'),
 ('quick', 'brown', 'fox'),
 ('brown', 'fox', 'jumps'),
 ('fox', 'jumps', 'over'),
 ('jumps', 'over', 'the'),
 ('over', 'the', 'lazy'),
 ('the', 'lazy', 'dog'),
 ('lazy', 'dog', 'The'),
 ('dog', 'The', 'quick'),
 ('The', 'quick', 'brown'),
 ('quick', 'brown', 'fox'),
 ('brown', 'fox', 'jumps'),
 ('fox', 'jumps', 'over'),
 ('jumps', 'over', 'the'),
 ('over', 'the', 'lazy'),
 ('the', 'lazy', 'dog'),
 ('lazy', 'dog', '<s>'),
 ('dog', '<s>', 'The'),
 ('<s>', 'The', 'Fulton'),
 ('The', 'Fulton', 'County'),
 ('Fulton', 'County', 'Grand'),
 ('County', 'Grand', 'Jury'),
 ('Grand', 'Jury', 'said'),
 ('Jury', 'said', 'Friday'),
 ('said', 'Friday', 'an'),
 ('Friday', 'an', 'investigation'),
 ('an', 'investigation', 'of'),
 ('investigation', 'of', "Atlanta's"),
 ('of', "Atlanta's", 'recent'),
 ("Atlanta's", 'recent', 'primary'),
 ('recent', 'primary', 'election'),
 ('primary', 'election', 'produced'),
 ('election', 'produced', '``'),
 ('produced', '``', 'no'),
 ('``', '

### Part ii

In [282]:
def calc_trigram(word1, word2, word3, trigrams):
    occurances=(Counter(trigrams))[(word1, word2, word3)]
    N_w1w2=(Counter(bigrams))[(word1, word2)]
    return (occurances+1)/(N_w1w2+V)

In [303]:
calc_trigram("This", "is", "a", trigrams)

0.00048648648648648646

### Part iii

In [299]:
def predict_next_tri(word1, word2, trigrams):
    tri_counter=Counter(trigrams)
    N_w1w2=Counter((w1, w2) for w1, w2, _ in trigrams)
    w1w2=N_w1w2[(word1, word2)]
    pred=None
    max_p=0
    for word3 in vocab:
        c_w1w2w3=tri_counter[(word1, word2, word3)]
        p=(c_w1w2w3+1)/(V+w1w2)
        if p>max_p:
            max_p=p
            pred=word3
    return pred

In [467]:
predict_next_tri("This", "is", trigrams)

'a'

### Part iv

In [306]:
def predict_sentence_tri(initial, trigrams, limit=5):
    words=initial.split()
    for i in range(limit-len(words)):
        next_word=predict_next_tri(words[-2], words[-1], trigrams)
        if not next_word:
            break
        words.append(next_word)
    return " ".join(words)

In [477]:
predict_sentence_tri("one of the United States", trigrams, 10)

'one of the United States , and the late Mrs.'

## Q4

In [315]:
bigrams_browns=get_bigrams(brown_corpus)

In [317]:
trigrams_browns=get_trigrams(brown_corpus)

#### Webtext corpus

In [715]:
all_words=[]

bigrams_webtext=get_bigrams(webtext_corpus)

# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

11379
152970


In [717]:
all_words=[]

trigrams_webtext=get_trigrams(webtext_corpus)

# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

11379
152970


In [719]:
calc_prob1(bigrams_webtext, "This", "is")

0.0013987236646560014

In [721]:
predict_next("this", bigrams_webtext)

'page'

In [819]:
predict_sentence("this is", bigrams_webtext, limit=7)

'this is not work in the page'

In [725]:
calc_trigram("This", "is", "the", trigrams_webtext)

0.0003506311360448808

In [727]:
predict_next_tri("This", "is", trigrams_webtext)

'the'

In [731]:
predict_sentence_tri("This is", trigrams_webtext, 10)

'This is the default browser " Advanced " tree opens'

In [356]:
webtext_corpus[0]

'<s> Cookie Manager : " Don \' t allow sites that set removed cookies to set future cookies " should stay checked When in full screen mode Pressing Ctrl - N should open a new browser when only download dialog is left open add icons to context menu So called " tab bar " should be made a proper toolbar or given the ability collapse / expand . </s>'

## Perplexity

### Bigram perplexity function

In [761]:
def uni_laplace(word, unigrams):
    w1=(Counter(unigrams))[word]
    return (w1+1)/(N+V)

In [763]:
def perplexity(sentence, bigrams, unigrams):
    words=sentence.split()
    ucounter=Counter(unigrams)
    vocab_size=len(ucounter)
    total_words=len(unigrams)
    product=uni_laplace(words[0], unigrams)
    for i in range(1,len(words)):
        p=calc_prob1(bigrams, words[i-1], words[i])
        product=product*p
    return product**(-1/len(words))

In [765]:
all_words=[]

bigrams_webtext=get_bigrams(webtext_corpus)


# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

11379
152970


In [767]:
for i in range(10):
    print(perplexity(reuters_corpus[i], bigrams_webtext, all_words))

7113.440045771614
5879.802410699634
3239.7807805651746
5697.519028267728
5639.622882060179
4374.971742578614
6451.410962793207
3188.8539901750105
1421.4493696368706
3984.5324934917103


In [769]:
all_words=[]

trigrams_webtext=get_trigrams(webtext_corpus)


# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

11379
152970


### Trigram perplexity function

In [771]:
def perplexity_trigrams(sentence, trigrams, bigrams, unigrams):
    words=sentence.split()
    product=uni_laplace(words[0], unigrams)*calc_prob1(bigrams, words[0], words[1])
    for i in range(2,len(words)):
        p=calc_trigram(words[i-2], words[i-1], words[i], trigrams)
        product=product*p
    return product**(-1/len(words))

In [773]:
for i in range(10):
    print(perplexity_trigrams(reuters_corpus[i], trigrams_webtext, bigrams_webtext, all_words))

9949.607249476796
8705.400880812345
8587.036869583866
9349.062059120739
9755.140306043542
6658.405889885502
9752.74707673824
5905.809156218562
5901.013910629062
8924.288095480939


### Calculations for the written report

### Q1

In [797]:
all_words=[]

bigrams_brown=get_bigrams(brown_corpus)

# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

15086
118731


In [799]:
perp_values=[]
for i in range(5):
    perp_values.append(perplexity(brown_corpus[i], bigrams_brown, all_words))

print(sum(perp_values)/len(perp_values))

2105.2210692456542


In [800]:
all_words=[]

trigrams_brown=get_trigrams(brown_corpus)

In [803]:
perp_values=[]
for i in range(5):
    perp_values.append(perplexity_trigrams(brown_corpus[i], trigrams_brown, bigrams_brown, all_words))

print(sum(perp_values)/len(perp_values))

4414.472328046814


### Q2

In [793]:
perp_values=[]
for i in range(25):
    perp_values.append(perplexity(reuters_corpus[i], bigrams_brown, all_words))

print(sum(perp_values)/len(perp_values))

5506.604062485168


In [794]:
perp_values=[]
for i in range(25):
    perp_values.append(perplexity_trigrams(reuters_corpus[i], trigrams_brown, bigrams_brown, all_words))

print(sum(perp_values)/len(perp_values))

10358.815828025346


In [783]:
all_words=[]

bigrams_webtext=get_bigrams(webtext_corpus)

# Size of vocabulary = V
vocab=Counter(all_words)
V=len(vocab)
print(V)
N=len(all_words)
print(N)

11379
152970


In [785]:
perp_values=[]
for i in range(25):
    perp_values.append(perplexity(reuters_corpus[i], bigrams_webtext, all_words))

print(sum(perp_values)/len(perp_values))

4823.552140736586


In [786]:
all_words=[]

trigrams_webtext=get_trigrams(webtext_corpus)

In [789]:
perp_values=[]
for i in range(25):
    perp_values.append(perplexity_trigrams(reuters_corpus[i], trigrams_webtext, bigrams_webtext, all_words))

print(sum(perp_values)/len(perp_values))

8331.805068795325
